In [1]:

# STEP 1: Setup Database & Insert VM Workloads


import sqlite3
import numpy as np
import random

# Create database
conn = sqlite3.connect("cloud_workloads.db")
cursor = conn.cursor()

# Drop old table if exists and create new one
cursor.execute("DROP TABLE IF EXISTS vm_traces")
cursor.execute("""
CREATE TABLE vm_traces (
    vm_id INTEGER,
    timestamp TEXT,
    cpu_usage REAL
)
""")

# Insert mock workload data (5 VMs × 10 timestamps)
for vm in range(5):
    for t in range(10):
        cpu = round(random.uniform(0.1, 0.9), 2)  # 10–90 % CPU usage
        cursor.execute("INSERT INTO vm_traces VALUES (?, ?, ?)", (vm, f"T{t}", cpu))

conn.commit()
print("Database created & workloads inserted successfully!")


Database created & workloads inserted successfully!


In [2]:

#  STEP 2: Cloud Environment


class Host:
    def __init__(self, host_id, capacity=1.0):
        self.host_id = host_id
        self.capacity = capacity
        self.load = 0.0

    def place_vm(self, cpu_demand):
        if self.load + cpu_demand <= self.capacity:
            self.load += cpu_demand
            return True
        return False

    def energy_consumption(self):
        # Linear power model
        return 50 + 150 * self.load

class CloudEnv:
    def __init__(self, n_hosts=3):
        self.hosts = [Host(i) for i in range(n_hosts)]

    def reset(self):
        for h in self.hosts:
            h.load = 0.0

    def total_energy(self):
        return sum(h.energy_consumption() for h in self.hosts)


In [3]:

# STEP 3: RL Agent (Q-Learning)
class RLAgent:
    def __init__(self, n_hosts, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.q_table = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.n_hosts = n_hosts

    def get_state(self, hosts):
        return tuple(round(h.load, 2) for h in hosts)

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_hosts - 1)
        return max(range(self.n_hosts), key=lambda a: self.q_table.get((state, a), 0))

    def update(self, state, action, reward, next_state):
        old_q = self.q_table.get((state, action), 0)
        max_next = max([self.q_table.get((next_state, a), 0)
                        for a in range(self.n_hosts)], default=0)
        new_q = old_q + self.alpha * (reward + self.gamma * max_next - old_q)
        self.q_table[(state, action)] = new_q


In [4]:

# STEP 4: Baseline Algorithm – PABFD


def pabfd_placement(env, workloads):
    env.reset()
    sorted_wl = sorted(workloads, key=lambda x: -x[1])
    for vm_id, cpu in sorted_wl:
        best_host = None
        best_energy = float("inf")
        for host in env.hosts:
            if host.load + cpu <= host.capacity:
                old_load = host.load
                host.load += cpu
                energy = env.total_energy()
                host.load = old_load
                if energy < best_energy:
                    best_energy = energy
                    best_host = host
        if best_host:
            best_host.place_vm(cpu)
    return [round(h.load, 2) for h in env.hosts], round(env.total_energy(), 2)


In [5]:

# STEP 5: Simulation Loop & Comparison


agent = RLAgent(n_hosts=3)
env = CloudEnv(n_hosts=3)

cursor.execute("SELECT vm_id, cpu_usage FROM vm_traces")
workloads = cursor.fetchall()

# Train RL agent
for episode in range(10):
    env.reset()
    total_reward = 0
    for vm_id, cpu in workloads:
        state = agent.get_state(env.hosts)
        action = agent.choose_action(state)
        success = env.hosts[action].place_vm(cpu)
        reward = -env.total_energy() if success else -500
        next_state = agent.get_state(env.hosts)
        agent.update(state, action, reward, next_state)
        total_reward += reward
    print(f"Episode {episode+1}, Reward: {total_reward:.2f}")

# Final RL placement
env.reset()
for vm_id, cpu in workloads:
    state = agent.get_state(env.hosts)
    action = agent.choose_action(state)
    env.hosts[action].place_vm(cpu)
rl_loads = [round(h.load, 2) for h in env.hosts]
rl_energy = round(env.total_energy(), 2)

# PABFD baseline
pabfd_loads, pabfd_energy = pabfd_placement(env, workloads)

print("\n===== Comparison =====")
print("RL Placement Loads:", rl_loads, "| Energy:", rl_energy, "W")
print("PABFD Placement Loads:", pabfd_loads, "| Energy:", pabfd_energy, "W")


Episode 1, Reward: -23931.50
Episode 2, Reward: -23786.50
Episode 3, Reward: -23710.00
Episode 4, Reward: -23629.50
Episode 5, Reward: -23746.50
Episode 6, Reward: -23683.00
Episode 7, Reward: -23731.00
Episode 8, Reward: -23743.50
Episode 9, Reward: -23678.50
Episode 10, Reward: -23541.00

===== Comparison =====
RL Placement Loads: [0.97, 0.21, 0.96] | Energy: 471.0 W
PABFD Placement Loads: [1.0, 1.0, 0.99] | Energy: 598.5 W
